# Loan Default Prediction

## Phase 2: Data Cleaning

### Objective

The objective of this notebook is to clean the dataset by handling missing values
and encoding categorical variables, so it's ready for model training.

In this notebook, we will:
- Handle missing values (numeric and categorical)
- Encode categorical variables into numeric form
- Drop non-predictive columns
- Save the cleaned dataset for use in the next phase

In [1]:
import pandas as pd

train = pd.read_csv("../data/train.csv")
train.shape

(614, 13)

### Drop Non-Predictive Columns

`Loan_ID` is a unique identifier for each applicant and has no predictive value,
so it is dropped before training.

In [2]:
train = train.drop('Loan_ID', axis=1)
train.shape

(614, 12)

### Handle Missing Values

- Categorical columns (Gender, Married, Dependents, Self_Employed) are filled with
  the mode (most frequent value), since we can't average text data.
- Numeric columns (LoanAmount, Loan_Amount_Term) are filled with the median, which
  is less sensitive to outliers than the mean.
- Credit_History is technically numeric but only takes values 0 or 1 (a flag), so
  it's filled with the mode rather than the median.

In [3]:
categorical_cols = ['Gender', 'Married', 'Dependents', 'Self_Employed']

for col in categorical_cols:
    train[col] = train[col].fillna(train[col].mode()[0])

train[categorical_cols].isnull().sum()

Gender           0
Married          0
Dependents       0
Self_Employed    0
dtype: int64

In [4]:
train['LoanAmount'] = train['LoanAmount'].fillna(train['LoanAmount'].median())
train['Loan_Amount_Term'] = train['Loan_Amount_Term'].fillna(train['Loan_Amount_Term'].median())
train['Credit_History'] = train['Credit_History'].fillna(train['Credit_History'].mode()[0])

train.isnull().sum()

Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64

### Encode Categorical Variables

- Binary columns (Gender, Married, Education, Self_Employed, Loan_Status) are mapped
  directly to 0/1.
- `Dependents` has a "3+" value which is converted to the number 3.
- `Property_Area` has 3 unrelated categories (Urban/Rural/Semiurban), so it's
  one-hot encoded instead of assigned arbitrary numbers — this avoids implying a
  false order between categories.

In [5]:
train['Gender'] = train['Gender'].map({'Male': 1, 'Female': 0})
train['Married'] = train['Married'].map({'Yes': 1, 'No': 0})
train['Education'] = train['Education'].map({'Graduate': 1, 'Not Graduate': 0})
train['Self_Employed'] = train['Self_Employed'].map({'Yes': 1, 'No': 0})
train['Loan_Status'] = train['Loan_Status'].map({'Y': 1, 'N': 0})

train.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,1,0,0,1,0,5849,0.0,128.0,360.0,1.0,Urban,1
1,1,1,1,1,0,4583,1508.0,128.0,360.0,1.0,Rural,0
2,1,1,0,1,1,3000,0.0,66.0,360.0,1.0,Urban,1
3,1,1,0,0,0,2583,2358.0,120.0,360.0,1.0,Urban,1
4,1,0,0,1,0,6000,0.0,141.0,360.0,1.0,Urban,1


In [6]:
train['Dependents'] = train['Dependents'].replace('3+', 3).astype(int)
train['Dependents'].unique()

array([0, 1, 2, 3])

In [7]:
train = pd.get_dummies(train, columns=['Property_Area'], drop_first=True)
train.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Loan_Status,Property_Area_Semiurban,Property_Area_Urban
0,1,0,0,1,0,5849,0.0,128.0,360.0,1.0,1,False,True
1,1,1,1,1,0,4583,1508.0,128.0,360.0,1.0,0,False,False
2,1,1,0,1,1,3000,0.0,66.0,360.0,1.0,1,False,True
3,1,1,0,0,0,2583,2358.0,120.0,360.0,1.0,1,False,True
4,1,0,0,1,0,6000,0.0,141.0,360.0,1.0,1,False,True


### Save Cleaned Dataset

The cleaned and encoded dataset is saved to the `data` folder, so it can be
used directly in the next phase (model training) without repeating this cleaning process.

In [10]:
train.to_csv('../data/cleaned_train.csv', index=False)
print("Saved cleaned_train.csv with shape:", train.shape)

Saved cleaned_train.csv with shape: (614, 13)


### Summary

- All missing values have been handled (0 remaining).
- All categorical variables have been encoded into numeric form.
- Non-predictive column (Loan_ID) has been removed.
- Cleaned dataset saved as `cleaned_train.csv`, ready for model training.